# Lesson 5.2 - Autoregressive & Diffusion Models (toy demos)

This notebook demonstrates both paradigms with lightweight runnable examples.

## Objectives

- Build a tiny autoregressive language model (n-gram) and sample text.
- Build a toy diffusion model on 2D data to visualize denoising.
- Optionally run a minimal `diffusers` text-to-image pipeline if environment supports it.

## Setup & Imports

In [ ]:
from __future__ import annotations

import math
import random
from collections import Counter, defaultdict
from typing import Iterable

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import make_moons

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Section A - Tiny Autoregressive Text Demo

We use a trigram language model with Laplace smoothing to illustrate autoregressive factorization:

$$
p(x_{1:T}) = \prod_{t=1}^{T} p(x_t\mid x_{<t}).
$$

In [ ]:
corpus = [
    "generative models learn data distributions",
    "autoregressive models predict the next token",
    "diffusion models denoise noisy samples step by step",
    "large language models are autoregressive transformers",
    "retrieval augmented generation improves grounding",
    "tool use extends language models with actions",
]


def tokenize(text: str) -> list[str]:
    return [tok.strip().lower() for tok in text.split() if tok.strip()]


START = "<s>"
END = "</s>"

vocab = set([START, END])
counts = defaultdict(Counter)

for line in corpus:
    toks = [START, START] + tokenize(line) + [END]
    vocab.update(toks)
    for i in range(2, len(toks)):
        ctx = (toks[i - 2], toks[i - 1])
        nxt = toks[i]
        counts[ctx][nxt] += 1

vocab = sorted(vocab)
V = len(vocab)


def next_token_distribution(context: tuple[str, str]) -> dict[str, float]:
    c = counts[context]
    total = sum(c.values())
    probs = {}
    for tok in vocab:
        probs[tok] = (c[tok] + 1.0) / (total + V)
    return probs


def sample_next(context: tuple[str, str], temperature: float = 1.0) -> str:
    probs = next_token_distribution(context)
    toks = np.array(list(probs.keys()))
    p = np.array(list(probs.values()), dtype=float)
    p = np.log(p + 1e-12) / max(temperature, 1e-6)
    p = np.exp(p - p.max())
    p = p / p.sum()
    return str(np.random.choice(toks, p=p))


def generate_text(max_len: int = 20, temperature: float = 1.0) -> str:
    w1, w2 = START, START
    out: list[str] = []
    for _ in range(max_len):
        nxt = sample_next((w1, w2), temperature=temperature)
        if nxt == END:
            break
        out.append(nxt)
        w1, w2 = w2, nxt
    return " ".join(out)


for t in [0.7, 1.0, 1.3]:
    samples = [generate_text(temperature=t) for _ in range(3)]
    print(f"temperature={t}:", samples)

## Section B - Toy Diffusion on 2D Data

We train a small noise-prediction network on two-moons points.

- Forward process adds Gaussian noise over timesteps.
- Model predicts noise $\epsilon_\theta(x_t, t)$.
- Reverse sampling iteratively denoises from random noise.

In [ ]:
x_np, _ = make_moons(n_samples=4000, noise=0.05, random_state=SEED)
x_np = x_np.astype(np.float32)
x = torch.tensor(x_np, device=device)

plt.figure(figsize=(5, 4))
plt.scatter(x_np[:, 0], x_np[:, 1], s=4, alpha=0.5)
plt.title("Training distribution (two moons)")
plt.show()

In [ ]:
T = 100
betas = torch.linspace(1e-4, 2e-2, T, device=device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)


def q_sample(x0: torch.Tensor, t: torch.Tensor, eps: torch.Tensor | None = None) -> tuple[torch.Tensor, torch.Tensor]:
    if eps is None:
        eps = torch.randn_like(x0)
    a_bar = alpha_bars[t].unsqueeze(1)
    xt = torch.sqrt(a_bar) * x0 + torch.sqrt(1.0 - a_bar) * eps
    return xt, eps


class TimeEmbedding(nn.Module):
    def __init__(self, dim: int = 32):
        super().__init__()
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(0, half, device=t.device).float() / max(half - 1, 1))
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)
        return emb


class EpsilonModel(nn.Module):
    def __init__(self, x_dim: int = 2, t_dim: int = 32, hidden: int = 128):
        super().__init__()
        self.t_embed = TimeEmbedding(t_dim)
        self.net = nn.Sequential(
            nn.Linear(x_dim + t_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, x_dim),
        )

    def forward(self, xt: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        t_norm = t.float() / (T - 1)
        t_emb = self.t_embed(t_norm)
        inp = torch.cat([xt, t_emb], dim=1)
        return self.net(inp)

In [ ]:
model = EpsilonModel().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

BATCH = 256
STEPS = 1200
losses = []

for step in range(1, STEPS + 1):
    idx = torch.randint(0, x.size(0), (BATCH,), device=device)
    x0 = x[idx]
    t = torch.randint(0, T, (BATCH,), device=device)

    xt, eps = q_sample(x0, t)
    eps_hat = model(xt, t)
    loss = F.mse_loss(eps_hat, eps)

    opt.zero_grad()
    loss.backward()
    opt.step()

    losses.append(loss.item())

plt.figure(figsize=(6, 3))
plt.plot(losses)
plt.title("Toy diffusion training loss")
plt.xlabel("step")
plt.ylabel("MSE")
plt.show()

In [ ]:
@torch.no_grad()
def p_sample_loop(model: nn.Module, n: int = 1000, capture_steps: Iterable[int] = (99, 70, 40, 10, 0)):
    model.eval()
    xt = torch.randn(n, 2, device=device)
    captured = {}

    for t_idx in reversed(range(T)):
        t = torch.full((n,), t_idx, device=device, dtype=torch.long)
        eps_theta = model(xt, t)

        alpha_t = alphas[t_idx]
        alpha_bar_t = alpha_bars[t_idx]
        beta_t = betas[t_idx]

        coef1 = 1.0 / torch.sqrt(alpha_t)
        coef2 = (1 - alpha_t) / torch.sqrt(1 - alpha_bar_t)
        mean = coef1 * (xt - coef2 * eps_theta)

        if t_idx > 0:
            z = torch.randn_like(xt)
            sigma = torch.sqrt(beta_t)
            xt = mean + sigma * z
        else:
            xt = mean

        if t_idx in capture_steps:
            captured[t_idx] = xt.detach().cpu().numpy()

    return xt.detach().cpu().numpy(), captured


samples_np, snapshots = p_sample_loop(model, n=2000)

plt.figure(figsize=(5, 4))
plt.scatter(samples_np[:, 0], samples_np[:, 1], s=4, alpha=0.5, label="generated")
plt.scatter(x_np[:1500, 0], x_np[:1500, 1], s=4, alpha=0.25, label="real")
plt.legend()
plt.title("Toy diffusion samples vs real data")
plt.show()

fig, axes = plt.subplots(1, len(snapshots), figsize=(14, 3))
for ax, (step, arr) in zip(axes, sorted(snapshots.items(), reverse=True)):
    ax.scatter(arr[:, 0], arr[:, 1], s=2, alpha=0.5)
    ax.set_title(f"t={step}")
plt.suptitle("Denoising trajectory snapshots")
plt.tight_layout()
plt.show()

## Section C - Optional `diffusers` Mini Demo

If dependencies, internet access, and compute are available, this cell attempts a tiny text-to-image generation run.

Set `RUN_DIFFUSERS_DEMO=1` in your environment to enable.

In [ ]:
import os

RUN_DIFFUSERS_DEMO = os.getenv("RUN_DIFFUSERS_DEMO", "0") == "1"

if RUN_DIFFUSERS_DEMO:
    try:
        from diffusers import StableDiffusionPipeline

        model_id = "hf-internal-testing/tiny-stable-diffusion-pipe"
        pipe = StableDiffusionPipeline.from_pretrained(model_id)
        pipe = pipe.to(device)
        out = pipe("a minimal line drawing of a robot", num_inference_steps=10)
        img = out.images[0]
        plt.figure(figsize=(4, 4))
        plt.imshow(img)
        plt.axis("off")
        plt.title("Optional diffusers output")
        plt.show()
    except Exception as exc:
        print(f"Diffusers demo skipped: {exc}")
else:
    print("Diffusers demo disabled. Set RUN_DIFFUSERS_DEMO=1 to try it.")

## Connect to Theory

- The n-gram demo reflects explicit conditional factorization in autoregressive modeling.
- The diffusion demo shows forward noising and reverse denoising loops.
- Compared with AR decoding, diffusion generation trades higher quality potential for multi-step sampling cost.

## Business Case Studies & Exceptions

### Case 1: Marketing Creative Pipeline

- **Use**: diffusion for image variants + autoregressive model for copy variants.
- **Benefit**: faster campaign iteration.
- **Exception**: high inference cost during traffic spikes; pre-generate and cache frequent templates.

### Case 2: Voice Assistant Content Generation

- **Use**: autoregressive text generation for responses; diffusion for voice style transfer in some stacks.
- **Benefit**: richer personalization.
- **Exception**: stricter latency budgets may force distilled or smaller models.

### Case 3: Safety and Abuse Risks

- **Risk**: generating misleading or policy-violating content.
- **Pattern**: moderation filters, watermarking strategy, audit logging, human review for high-risk outputs.

## Interview Questions & Answers

1. **What is an autoregressive model?**  
   A model that generates one step at a time using conditional probabilities on prior generated context.

2. **How do autoregressive models train?**  
   Usually via maximum likelihood with teacher forcing over true previous tokens.

3. **What is the intuition behind diffusion models?**  
   Learn to reverse a process that progressively adds noise to data.

4. **Why are diffusion models slower at inference?**  
   They require many denoising iterations per sample.

5. **What does the diffusion model typically predict?**  
   The noise component added at each timestep (or equivalent parameterization).

6. **What is a noise schedule?**  
   A sequence controlling how much noise is added at each forward step.

7. **When would you prefer autoregressive models?**  
   Long text/code generation with strong sequential dependency and mature tooling.

8. **When would you prefer diffusion models?**  
   High-fidelity image generation or multimodal synthesis tasks prioritizing quality.

9. **What is a practical way to reduce diffusion latency?**  
   Use fewer-step samplers, latent-space generation, model distillation, and hardware optimization.

10. **How do sampling settings affect AR outputs?**  
   Temperature/top-k/top-p tune diversity versus determinism.

11. **Can diffusion models overfit?**  
   Yes; watch for memorization and near-duplicate outputs.

12. **How do you compare AR vs diffusion for production?**  
   Evaluate quality, latency, throughput, cost, and risk controls against product requirements.